# Homework 1 — Agentic RAG

LLM Zoomcamp 2026 · Cohort

Building a RAG system over the course lesson pages, then making it agentic.

**Knowledge base:** lesson markdown files from the course repo at commit `8c1834d`

## Setup

In [1]:
import os
import subprocess
from pathlib import Path
from openai import OpenAI
import minsearch
from gitsource import GithubRepositoryDataReader, chunk_documents


def _load_api_key_from_env_file(env_file: Path) -> str | None:
    """Parse OPENAI_API_KEY from a .env file. Returns None if not found."""
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line.startswith("OPENAI_API_KEY=") and not line.startswith("#"):
            return line.split("=", 1)[1].strip()
    return None


if not os.environ.get("OPENAI_API_KEY"):
    try:
        os.environ["OPENAI_API_KEY"] = subprocess.check_output(
            ["op", "read", "op://Personal/OpenAIDataTalk/credential"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        # Fallback: read from .env file in the project root
        env_file = Path("../.env")
        key = _load_api_key_from_env_file(env_file) if env_file.exists() else None
        if key:
            os.environ["OPENAI_API_KEY"] = key
        else:
            raise EnvironmentError(
                "OPENAI_API_KEY not found. "
                "Either run 1Password ('op signin') or create a ../.env file with OPENAI_API_KEY=<your-key>"
            )

client = OpenAI()

## Preparation — Fetch lesson pages from GitHub

Using `gitsource` to download the course repo at a fixed commit for reproducibility.

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages?

In [5]:
print(f"Number of lesson pages: {len(documents)}")

Number of lesson pages: 72


## Q2. Indexing and searching

Build a minsearch index and search for the first result's filename.

In [6]:
index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)

print(f"First result filename: {results[0]['filename']}")

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG — input tokens with full documents

`LessonRAG` adapts `RAGBase` to the lesson schema and exposes token usage.

In [8]:
import sys
sys.path.insert(0, "/Users/carlosibarra/projects/llm-zoomcamp/01-agentic-rag/code")
from rag_helper import RAGBase, INSTRUCTIONS, PROMPT_TEMPLATE


class LessonRAG(RAGBase):

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"FILENAME: {doc['filename']}")
            lines.append(f"CONTENT: {doc['content']}")
            lines.append("")
        return "\n".join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt},
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages,
        )
        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        answer = response.output_text
        input_tokens = response.usage.input_tokens
        return answer, input_tokens

In [9]:
rag = LessonRAG(index=index, llm_client=client, model="gpt-4o-mini")

query = "How does the agentic loop keep calling the model until it stops?"
answer, input_tokens = rag.rag(query)

print(f"Input tokens: {input_tokens}")
print(f"\nAnswer:\n{answer}")

Input tokens: 7142

Answer:
The agentic loop keeps calling the model until it stops by employing a `while` loop that continuously checks for function calls. Here's how it works:

1. The loop begins by sending the current message history to the model.
2. If the model's response includes a function call (indicating it needs to perform an action, like a search), the loop executes that function and appends the result to the message history.
3. The loop increments an iteration counter to track how many rounds of requests have occurred.
4. This process repeats until the model returns a response that does not contain any function calls. At this point, it indicates that the model has finished processing and is ready to provide a final answer.

The exit condition for the loop is simply if the model's response does not include any further function calls, at which point it breaks out of the loop.


## Q4. Chunking — how many chunks?

In [10]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


## Q5. RAG with chunking — fewer input tokens?

In [11]:
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

rag_chunked = LessonRAG(index=chunk_index, llm_client=client, model="gpt-4o-mini")

answer_chunked, input_tokens_chunked = rag_chunked.rag(query)

print(f"Input tokens (full docs):  {input_tokens}")
print(f"Input tokens (chunked):    {input_tokens_chunked}")
print(f"Ratio: {input_tokens / input_tokens_chunked:.1f}x fewer tokens with chunking")

Input tokens (full docs):  7142
Input tokens (chunked):    2325
Ratio: 3.1x fewer tokens with chunking


## Q6. Turning it into an agent

Using `toyaikit` — the LLM decides when and how many times to call `search`.

In [12]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

agent_instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

search_call_count = 0

def search(query: str) -> list[dict]:
    """Search the course lesson pages for content matching the query."""
    global search_call_count
    search_call_count += 1
    return chunk_index.search(query, num_results=5)


agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=agent_instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-4o-mini"),
)

In [13]:
search_call_count = 0

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

print(f"\nNumber of search() calls: {search_call_count}")

-> Response received


-> Response received



Number of search() calls: 3
